# Surface-density fly-through

Camera path over one snapshot; perspective surface-density map per frame. Example paths: inclined circular orbit, orbit with radial drift.


In [ ]:
from pathlib import Path

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.insert(0, str(Path("..").resolve()))

from simviz.field_plots import project_surface_density_camera
from simviz.projections import rotate_about_axis, rotate_to_bar_frame
from simviz.utils import read_snapshot_hdf5

EXAMPLE_OUT = Path("../example_output")
EXAMPLE_OUT.mkdir(exist_ok=True)

print("imports done")

In [ ]:
# Gas snapshot: positions + masses only (MHD not required).
snap_path = Path("/Users/hph/Research/Archive/Old_Code/arepo_CMZ/TS_2020/bar_sink_sne_250.hdf5")

data, header = read_snapshot_hdf5(snap_path)
print(f"Loaded: {snap_path}")
print(f"N gas cells: {len(data['Coordinates']):,}")
print("PartType0 fields:", sorted(data.keys()))

In [ ]:
# GC-center; bar frame 
t = float(header["Time"])
box = float(header["BoxSize"])

x, y, z = np.asarray(data["Coordinates"], dtype=np.float64).T
masses = np.asarray(data["Masses"], dtype=np.float64)

x -= box / 2.0
y -= box / 2.0
z -= box / 2.0

zeros = np.zeros_like(x)
x, y, _, _ = rotate_to_bar_frame(x, y, zeros, zeros, t)

print(f"Prepared {x.size:,} particles in bar frame")

In [ ]:
def inclined_orbit_path(
    n_frames,
    radius=8.0,
    tilt_axis=(1.0, 1.0, 0.3),
    tilt_deg=35.0,
    z_offset=0.0,
    center=(0.0, 0.0, 0.0),
):
    #Camera path: circular orbit rotated about a non-trivial axis.
    theta = np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False)
    base = np.column_stack([
        radius * np.cos(theta),
        radius * np.sin(theta),
        np.full(n_frames, z_offset),
    ])
    tilted = rotate_about_axis(base, axis=tilt_axis, angle=np.radians(tilt_deg))
    return tilted + np.asarray(center, dtype=np.float64)


def orbit_with_radial_drift_path(
    n_frames,
    r_start=12.0,
    r_end=6.0,
    n_turns=1.5,
    tilt_axis=(1.0, 1.0, 0.3),
    tilt_deg=35.0,
    center=(0.0, 0.0, 0.0),
):
    #Camera path: orbit plus radial in/out drift.
    theta = np.linspace(0.0, 2.0 * np.pi * n_turns, n_frames, endpoint=False)
    radius = np.linspace(r_start, r_end, n_frames)
    base = np.column_stack([
        radius * np.cos(theta),
        radius * np.sin(theta),
        np.zeros(n_frames),
    ])
    tilted = rotate_about_axis(base, axis=tilt_axis, angle=np.radians(tilt_deg))
    return tilted + np.asarray(center, dtype=np.float64)

In [ ]:
def render_surface_density_frame(
    x,
    y,
    z,
    masses,
    camera_position,
    target=(0.0, 0.0, 0.0),
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=55.0,
    nx=700,
    ny=700,
    z_near=0.2,
    z_far=40.0,
    vmin=1e-3,
    vmax=2e1,
    cmap="Greys",
    ax=None,
):
    sigma, extent = project_surface_density_camera(
        x,
        y,
        z,
        masses,
        camera_position=camera_position,
        target=target,
        up_hint=up_hint,
        fov_x_deg=fov_x_deg,
        nx=nx,
        ny=ny,
        z_near=z_near,
        z_far=z_far,
    )

    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(6, 6), dpi=150)
    else:
        fig = ax.figure

    im = ax.imshow(
        sigma,
        origin="lower",
        extent=extent,
        norm=colors.LogNorm(vmin=vmin, vmax=vmax),
        cmap=cmap,
        interpolation="nearest",
    )
    ax.set_xlabel("camera x")
    ax.set_ylabel("camera y")
    return fig, ax, im

In [ ]:
# Single-frame preview
path = inclined_orbit_path(n_frames=120, radius=10.0, tilt_deg=35.0)

i = 0
cam = path[i]
fig, ax, im = render_surface_density_frame(
    x,
    y,
    z,
    masses,
    camera_position=cam,
    target=(0.0, 0.0, 0.0),
    up_hint=(0.0, 0.0, 1.0),
    fov_x_deg=55.0,
    z_near=0.2,
    z_far=30.0,
    vmin=5e-4,
    vmax=2e1,
)
fig.colorbar(im, ax=ax, pad=0.02, label=r"Surface density [code mass / pixel]")
ax.set_title(f"Preview frame {i}: cam=({cam[0]:.2f}, {cam[1]:.2f}, {cam[2]:.2f})")
fig.savefig(EXAMPLE_OUT / "surface_density_preview.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# PNG sequence (encode video separately)
out_dir = Path("flythrough_frames")
out_dir.mkdir(exist_ok=True)

# Alternate camera paths:
# path = inclined_orbit_path(n_frames=180, radius=10.0, tilt_deg=35.0)
path = orbit_with_radial_drift_path(n_frames=180, r_start=12.0, r_end=6.0, n_turns=1.5)

for i, cam in enumerate(path):
    fig, ax, im = render_surface_density_frame(
        x,
        y,
        z,
        masses,
        camera_position=cam,
        target=(0.0, 0.0, 0.0),
        up_hint=(0.0, 0.0, 1.0),
        fov_x_deg=55.0,
        z_near=0.2,
        z_far=30.0,
        vmin=5e-4,
        vmax=2e1,
    )
    ax.set_title(f"frame {i:04d}")
    fig.savefig(out_dir / f"frame_{i:04d}.png", bbox_inches="tight")
    plt.close(fig)

print(f"Wrote {len(path)} frames to: {out_dir.resolve()}")

In [ ]:
# MP4: run from this directory (same folder as flythrough_frames/). libx264+yuv420p needs even width and height; bbox_inches='tight' can produce odd sizes, so crop to even pixels first:
# ffmpeg -y -framerate 30 -i flythrough_frames/frame_%04d.png -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" -c:v libx264 -pix_fmt yuv420p surface_density_flythrough.mp4
